# meikey を Google Colab の GPU で動かす

[meikey](https://github.com/waxsd100/meikey) は、公開鍵に好きな文字列が入った ed25519 の SSH 鍵を探すツールです。
このノートブックでは、Colab の GPU（CUDA）と CPU で鍵を探します。上のセルから順に実行してください。
GPU・CPU・GPU + CPU のどれで探すかは、手順 5・6 のフォームで選べます。

## GPU の選び方

メニューの **ランタイム → ランタイムのタイプを変更 → ハードウェア アクセラレータ** で GPU を選びます。
この計算は 32 ビット整数の掛け算が中心なので、整数演算の多い GPU ほど速い見込みです（目安。実際の速さは手順 5 で測れます）:

**RTX PRO 6000（G4） > H100 > A100 > L4 > T4**

- どの GPU が使えるかは、その時の空き状況と契約しているプランで決まります。Google AI Ultra ではプレミアム GPU が使えます
- Google AI Ultra では **バックグラウンド実行** が使えるので、長い探索を始めたらブラウザのタブを閉じても続きます（最大 24 時間）
- CPU は AVX-512 IFMA・AVX-512・AVX2 のうち使えるものを自動で選びます。どれが使えるかは手順 1 と手順 4 で表示します
- 起動中は GPU を使っていなくてもコンピューティング ユニットが減るので、使い終わったら **ランタイム → ランタイムを接続解除して削除** してください

## 秘密鍵の扱い

秘密鍵は Colab の VM（Google のサーバー）の上で作られます。

- 手順 7 でダウンロードしたら、手順 8 で VM から消してください
- 秘密鍵にはパスフレーズが付いていません。手元で `ssh-keygen -p -f <秘密鍵>` を実行して、パスフレーズを付けてから使ってください

## 1. GPU と CPU を確認する

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total,driver_version --format=csv || echo "GPU なし"
!echo "CPU: $(nproc) コア / $(lscpu | sed -n 's/^Model name: *//p')"
!grep -q -w avx512ifma /proc/cpuinfo && echo "AVX-512 IFMA あり" || echo "AVX-512 IFMA なし"

## 2. Rust を入れる

meikey が使う Rust（1.99）は、ビルドのときに自動で入ります。

In [ ]:
import os, shutil

CARGO_BIN = os.path.expanduser("~/.cargo/bin")
if not os.path.exists(os.path.join(CARGO_BIN, "cargo")):
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal --default-toolchain none
if CARGO_BIN not in os.environ["PATH"]:
    os.environ["PATH"] = CARGO_BIN + ":" + os.environ["PATH"]
!rustup --version

## 3. meikey を取得してビルドする

初回は数分かかります。

In [ ]:
REPO = "https://github.com/waxsd100/meikey"
BRANCH = "master"

if os.path.exists("/content/meikey"):
    !git -C /content/meikey pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO} /content/meikey
%cd /content/meikey
# rust-toolchain.toml に書いたバージョン（1.99）を入れる
!rustup toolchain install
!RUSTFLAGS="-C target-cpu=native" cargo build --release
BIN = "/content/meikey/target/release/meikey"

## 4. GPU と CPU の状態を確かめる

GPU 向けのプログラムは meikey に埋め込んであり、NVIDIA のドライバだけで動きます（初回はドライバの変換に数秒かかります）。
NVRTC（libnvrtc）は、ドライバが古くて埋め込んだプログラムを読めないときの予備なので、見つかれば使えるようにしておきます。

In [ ]:
import glob

# libnvrtc.so を探して、見つけたものを LD_LIBRARY_PATH の先頭のディレクトリにリンクしておく
candidates = sorted(
    p for d in ["/usr/local/cuda/lib64", "/usr/local/cuda/targets/x86_64-linux/lib", "/usr/lib/x86_64-linux-gnu"]
              + glob.glob("/usr/local/lib/python3*/dist-packages/nvidia/cuda_nvrtc/lib")
              + glob.glob("/usr/local/lib/python3*/dist-packages/nvidia/cu*/lib")
    for p in glob.glob(d + "/libnvrtc.so*")
    if "builtins" not in p and ".alt." not in p
)
print("libnvrtc の候補:", candidates)
LIBDIR = "/content/meikey-libs"
os.makedirs(LIBDIR, exist_ok=True)
if candidates:
    nvrtc = candidates[-1]
    link = os.path.join(LIBDIR, "libnvrtc.so")
    if os.path.lexists(link):
        os.remove(link)
    os.symlink(nvrtc, link)
    # libnvrtc が読み込む libnvrtc-builtins も同じディレクトリから探せるように
    paths = [LIBDIR, os.path.dirname(nvrtc)]
    old = os.environ.get("LD_LIBRARY_PATH", "")
    os.environ["LD_LIBRARY_PATH"] = ":".join(paths + [p for p in old.split(":") if p and p not in paths])
else:
    print("libnvrtc は見つからなかった（無くても、埋め込んだプログラムをドライバが読めれば動く）")


import subprocess, codecs

def meikey(*args):
    # meikey を実行して、出力をそのまま（進捗の行の上書きも含めて）表示する
    p = subprocess.Popen([BIN, *map(str, args)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    dec = codecs.getincrementaldecoder("utf-8")("replace")
    while chunk := os.read(p.stdout.fileno(), 4096):
        print(dec.decode(chunk), end="", flush=True)
    return p.wait()

# 探すのに使うもの（手順 5・6 のフォームの選択肢）と、meikey に付ける引数
DEVICES = {"GPU": ["--gpu"], "CPU": [], "GPU + CPU": ["--both"]}

if meikey("--check", "--both") != 0:
    print("GPU は使えない。CPU だけで探せる:")
    meikey("--check")

### （任意）GPU で計算が正しいかのテスト

GPU で 8,000 鍵ほどを計算して ed25519-dalek と突き合わせ、実際に探して見つかることを確かめます（テスト用のビルドに数分かかります）。

In [ ]:
!MEIKEY_REQUIRE_SIMD=cuda cargo test --release --lib service::gpu

## 5. 速さを測る

入れたい文字列が見つかるまでの期待時間を測ります。「全部」なら GPU・CPU・GPU + CPU を順に測って比べます。

In [ ]:
TARGET = "wakokara"  #@param {type:"string"}
SECONDS = 10  #@param {type:"integer"}
DEVICE = "全部"  #@param ["全部", "GPU", "CPU", "GPU + CPU"]

for name in (DEVICES if DEVICE == "全部" else [DEVICE]):
    print(f"---- {name}")
    meikey(*DEVICES[name], "--bench", SECONDS, TARGET)

## 6. 鍵を探す

見つかると、`/content/keys/` に秘密鍵（`id_ed25519_<文字列>`）と公開鍵（`.pub`）を書き出します。
大小文字を区別すると、英字 1 文字ごとに 2 倍時間がかかります。
DEVICE で探すのに使うものを選びます。GPU + CPU は、CPU の 1 コアを GPU の操作に空けて残りで探し、どちらかが見つけたら止まります。

In [ ]:
TARGET = "wakokara"  #@param {type:"string"}
CASE_SENSITIVE = False  #@param {type:"boolean"}
COMMENT = ""  #@param {type:"string"}
DEVICE = "GPU + CPU"  #@param ["GPU", "CPU", "GPU + CPU"]

os.makedirs("/content/keys", exist_ok=True)
OUTPUT = f"/content/keys/id_ed25519_{TARGET}"
args = [*DEVICES[DEVICE], TARGET, "-o", OUTPUT, "-C", COMMENT]
if CASE_SENSITIVE:
    args.append("--case-sensitive")
meikey(*args)

## 7. 鍵をダウンロードする

In [ ]:
from google.colab import files

files.download(OUTPUT)
files.download(OUTPUT + ".pub")

## 8. VM から鍵を消す

ダウンロードを確かめてから実行してください。

In [ ]:
!shred -u {OUTPUT} && rm -f {OUTPUT}.pub && ls -la /content/keys